# Citrus: generate 50,000 draft examples
This notebook generates data; it does not train an adapter. Target: 15,000 educational knowledge pairs and 35,000 practical task pairs. Use a Kaggle CUDA GPU and enable Internet for the first download. There is no paid inference API. Runtime availability and Kaggle quotas may require multiple sessions. Completion is not guaranteed in one session.

Outputs remain drafts. Review factual accuracy, semantic duplication and code before training. The generator's schema checks are not quality approval.


In [ ]:
%pip install -q "transformers>=4.51,<5" "accelerate>=1,<2" "bitsandbytes>=0.45,<1" sentencepiece


In [ ]:
import pathlib, urllib.request
# Fixed repository revision makes the notebook reproducible.
REVISION = "8f0b921c6c64b1781615021bd7ab0cc02e01df22"
BASE = f"https://raw.githubusercontent.com/gentle58715-ship-it/Citrus-Training-/{REVISION}"
ROOT = pathlib.Path("/kaggle/working/citrus-generator")
ROOT.mkdir(exist_ok=True)
for name in ["generate_citrus_kaggle.py", "audit_citrus_v3.py"]:
    urllib.request.urlretrieve(f"{BASE}/scripts/{name}", ROOT / name)


In [ ]:
import sqlite3
OUT = pathlib.Path("/kaggle/working/citrus-50k")
OUT.mkdir(exist_ok=True)
# To resume, attach the previous output as a Kaggle input and put its DB path here.
RESUME_DB = ""
if RESUME_DB:
    if (OUT / "progress.sqlite").exists():
        raise RuntimeError("Output DB already exists; do not overwrite ongoing work")
    src = sqlite3.connect(f"file:{RESUME_DB}?mode=ro", uri=True)
    dst = sqlite3.connect(OUT / "progress.sqlite")
    src.backup(dst)
    dst.close(); src.close()


In [ ]:
import subprocess
# Stop after this session budget and resume from progress.sqlite later.
MAX_MINUTES = 480
try:
    subprocess.run(["python3", "-u", str(ROOT / "generate_citrus_kaggle.py"),
                    "--out", str(OUT), "--knowledge", "15000", "--practice", "35000",
                    "--max-minutes", str(MAX_MINUTES)], check=True)
finally:
    # Rebuild the export from committed SQLite rows even after an interrupted run.
    subprocess.run(["python3", str(ROOT / "generate_citrus_kaggle.py"),
                    "--out", str(OUT), "--export-only"], check=True)


In [ ]:
import json, shutil
manifest = json.loads((OUT / "manifest.json").read_text())
print(json.dumps(manifest, ensure_ascii=False, indent=2))
subprocess.run(["python3", str(ROOT / "audit_citrus_v3.py"), "--root", str(OUT)], check=True)
archive = shutil.make_archive("/kaggle/working/Citrus-50k-progress", "zip", OUT)
print("Saved:", archive)


## Continue or review
Save the notebook output and progress archive after each session. Attach it as an input to a new session, set RESUME_DB to the attached progress.sqlite path, and run again. The SQLite database is the authoritative checkpoint; do not edit it manually. The generator stops exactly at the configured 15,000/35,000 draft totals if it can continue receiving valid distinct outputs.

A 50,000-record manifest with status draft_generation_complete means generation only. It does not mean verified training data. All examples have review_status=draft and runtime_verified=false. The audit exports approved records only. Assign review decisions after actual review; do not bulk-approve to bypass checks. Related examples share category/topic family IDs to reduce train/eval overlap. This does not replace semantic deduplication or an independent benchmark.
